# 02 · Calidad de datos: duplicados, fuentes y columnas pendientes

Continúa el perfilado de `01_exploracion_inicial.ipynb`. Allí vimos que deduplicar por
`company + title + location` eliminaría **casi la mitad de las filas** (11,46 M → 5,79 M) y que
el tamaño de los grupos tenía una **joroba entre 6 y 11 filas**, con pico en 9.

Este notebook responde **de dónde sale esa joroba**, define la **regla de deduplicación** y
revisa las columnas que faltaban (ubicación, URL, cierres, salario, skills).
Al final hay un **resumen de decisiones** para el script de limpieza.

## Preparación

En vez de `duckdb.read_csv(...)` (que vuelve a leer el CSV de 2,9 GB en cada consulta),
cargamos el CSV **una sola vez** en una tabla en memoria con `CREATE TABLE`. Tarda un poco al
principio, pero luego cada consulta va mucho más rápida.

In [1]:
import duckdb
import pandas as pd

from radar_junior.config import RAW_DIR

pd.set_option("display.max_colwidth", 80)

duckdb.sql(f"""
    CREATE OR REPLACE TABLE jobs AS
    SELECT * FROM read_csv('{RAW_DIR / 'jobs.csv'}', types = {{'posted': 'VARCHAR'}})
""")
duckdb.sql("SELECT count(*) AS filas FROM jobs").df()

,filas
0,11462654


## 1. ¿Qué hay dentro de un grupo de 9 filas?

Cogemos un grupo de 9 filas de la fuente `shard` (la sección 2 explica por qué esa fuente).
`HAVING` filtra **después** de agrupar, por eso sirve para condiciones sobre `count(*)`.
El `ORDER BY` hace que salga siempre el mismo grupo cada vez que se ejecuta.

In [2]:
grupo = duckdb.sql("""
    SELECT company, title, location
    FROM jobs
    WHERE source = 'shard'
    GROUP BY company, title, location
    HAVING count(*) = 9
    ORDER BY company, title, location
    LIMIT 1
""").df()
grupo

,company,title,location
0,!K7 Music,Customer Service Representative,"Berlin, Berlin, Germany"


In [3]:
c, t, l = grupo.iloc[0]
duckdb.execute("""
    SELECT source, url, first_seen, closed_at, years_experience, level
    FROM jobs
    WHERE company = ? AND title = ? AND location = ?
    ORDER BY first_seen
""", [c, t, l]).df()

,source,url,first_seen,closed_at,years_experience,level
0,shard,https://k7-music-gmbh.jobs.personio.com/job/13e8407eeef3,2026-10-20,NaT,<NA>,mid
1,shard,https://k7-music-gmbh.jobs.personio.com/job/9b233089aa41,2026-10-22,NaT,<NA>,mid
2,shard,https://k7-music-gmbh.jobs.personio.com/job/84a4de42a763,2026-10-22,NaT,<NA>,mid
3,shard,https://k7-music-gmbh.jobs.personio.com/job/f9eaad5d0fc7,2026-10-26,NaT,<NA>,mid
4,shard,https://k7-music-gmbh.jobs.personio.com/job/ffc2c88578df,2026-10-28,NaT,<NA>,mid
5,shard,https://k7-music-gmbh.jobs.personio.com/job/f1ea040a6721,2026-10-30,NaT,<NA>,mid
6,shard,https://k7-music-gmbh.jobs.personio.com/job/40bb587c0deb,2026-11-01,NaT,<NA>,mid
7,shard,https://k7-music-gmbh.jobs.personio.com/job/c594552140fc,2026-11-01,NaT,<NA>,mid
8,shard,https://k7-music-gmbh.jobs.personio.com/job/f14e52111e83,2026-11-03,NaT,<NA>,mid


**Lectura:** las 9 filas son la misma oferta. Solo cambian la `url` y `first_seen`, que aparece
**cada pocos días**, y **ninguna está cerrada**. No es una empresa que republica (como Gopuff en
Birmingham, donde una se cerraba y otra se abría): son 9 copias abiertas a la vez.

> Los `?` son **parámetros**: DuckDB sustituye cada `?` por el valor de la lista, en orden.
> Es más seguro que meter el texto con un f-string (los títulos pueden tener comillas).

## 2. ¿La joroba es cosa de una fuente?

`count(*) OVER (PARTITION BY ...)` es una **función de ventana**: calcula el tamaño del grupo
pero **sin juntar las filas**, así que cada fila sabe cuántas hermanas tiene. Luego medimos,
por fuente, qué porcentaje de filas cae en grupos de 6 a 11.

In [4]:
duckdb.sql("""
    WITH con_tamano AS (
        SELECT source,
               count(*) OVER (PARTITION BY company, title, location) AS n
        FROM jobs
    )
    SELECT source,
           count(*) AS filas,
           round(100 * avg((n BETWEEN 6 AND 11)::INT), 1) AS pct_en_joroba,
           round(100 * avg((n = 1)::INT), 1) AS pct_unicas
    FROM con_tamano
    GROUP BY source
    ORDER BY filas DESC
    LIMIT 10
""").df()

,source,filas,pct_en_joroba,pct_unicas
0,shard,6501369,90.8,5.0
1,workday,1209235,0.1,95.6
2,eu,733803,0.0,98.8
3,oracle,382656,0.4,85.4
4,smartrecruiters,343627,0.1,82.6
5,adp,284577,0.2,92.4
6,greenhouse,228228,0.2,84.7
7,ultipro,170709,0.9,78.2
8,paycom,148664,0.5,89.4
9,dayforce,146326,3.2,77.5


**Lectura:** `shard` es el **57 % del dataset** (6,5 M filas) y el **91 %** de sus filas está en
la joroba. En el resto de fuentes la joroba es < 1 % y la gran mayoría de filas son únicas.

## 3. ¿Cómo funciona `shard`?

Para cada oferta de `shard`, `lag()` trae el `first_seen` de la copia anterior; la diferencia
en días es la separación entre copias.

In [5]:
duckdb.sql("""
    WITH copias AS (
        SELECT datediff('day',
                        lag(first_seen) OVER (PARTITION BY company, title, location
                                              ORDER BY first_seen),
                        first_seen) AS dias_desde_anterior
        FROM jobs
        WHERE source = 'shard'
    )
    SELECT dias_desde_anterior, count(*) AS copias
    FROM copias
    WHERE dias_desde_anterior IS NOT NULL
    GROUP BY 1
    ORDER BY copias DESC
    LIMIT 6
""").df()

,dias_desde_anterior,copias
0,2,3400942
1,0,1117482
2,1,524046
3,4,247941
4,3,32258
5,8,21120


In [6]:
duckdb.sql("""
    SELECT source = 'shard' AS es_shard,
           count(*) AS filas,
           min(first_seen) AS desde,
           max(first_seen) AS hasta,
           round(100 * avg((closed_at IS NOT NULL)::INT), 1) AS pct_cerradas,
           round(100 * avg((ghost_score IS NULL)::INT), 1) AS pct_ghost_nulo
    FROM jobs
    GROUP BY 1
""").df()

,es_shard,filas,desde,hasta,pct_cerradas,pct_ghost_nulo
0,True,6501369,2020-01-01,2026-11-04,0.0,0.0
1,False,4961285,2026-05-24,2026-11-22,32.9,32.9


**Lectura:**
- `shard` hace una **foto cada 2 días** y en cada foto crea una fila nueva con URL distinta.
- `shard` **nunca registra cierres** (0 % de `closed_at`). Por tanto, en `shard` "`closed_at`
  vacío" **no significa "sigue abierta"**: significa "no lo sabemos".
- En las demás fuentes, `ghost_score` está vacío exactamente en las ofertas cerradas
  (32,9 % y 32,9 %): **solo se calcula para ofertas abiertas**.
- `shard` acaba el **4 de noviembre**; las demás fuentes llegan hasta el 22.

## 4. Las filas antiguas (2020-2023)

In [7]:
duckdb.sql("""
    SELECT source, count(*) AS filas,
           count(url) AS con_url,
           count(location) AS con_location,
           count(DISTINCT first_seen) AS fechas_distintas
    FROM jobs
    WHERE first_seen < '2026-01-01'
    GROUP BY source
""").df()

,source,filas,con_url,con_location,fechas_distintas
0,shard,123046,0,0,45


In [8]:
duckdb.sql("""
    SELECT company, title, location, first_seen
    FROM jobs
    WHERE first_seen < '2026-01-01'
    LIMIT 5
""").df()

,company,title,location,first_seen
0,"""Coelix"".",NodeJS developer,None,2020-01-01
1,"""In Diverse Company""",Junior QA Engineer,None,2020-02-01
2,"""In Diverse Company""",Middle Python Backend Engineer,None,2020-02-01
3,"""Альфа-Банк Україна""",Java Atlassian Developer (Jira),None,2021-09-01
4,.doCode,Back - End Developer (AWS),None,2023-01-01


**Lectura:** 123.046 filas de `shard` con fechas de 2020 a 2023, siempre el **día 1 del mes**,
**sin URL ni ubicación**. Parece un archivo histórico de ofertas tech (muchas empresas
ucranianas). No describen el mercado actual → **se excluyen**.

## 5. ¿`shard` repite ofertas de las otras fuentes?

Si `shard` solo copiara lo que ya traen Workday, Greenhouse, etc., podríamos tirarlo entero.
Lo medimos con un `JOIN` entre las combinaciones únicas de cada lado.

In [9]:
duckdb.sql("""
    WITH s AS (SELECT DISTINCT company, title, location FROM jobs WHERE source = 'shard'),
         o AS (SELECT DISTINCT company, title, location FROM jobs WHERE source <> 'shard')
    SELECT (SELECT count(*) FROM s) AS ofertas_shard,
           (SELECT count(*) FROM s JOIN o USING (company, title, location)) AS tambien_en_otras
""").df()

,ofertas_shard,tambien_en_otras
0,1097074,72642


**Lectura:** solo el **6,6 %** de las ofertas de `shard` aparece también en otras fuentes.
`shard` aporta ofertas propias → **se conserva, pero deduplicado**.

## 6. Regla de deduplicación y su efecto

**Regla:**
- Fuentes distintas de `shard`: se quedan como están (sus repeticiones son pocas y muchas son
  plazas reales en tiendas distintas, como vimos con Gopuff y *Client Careers Site*).
- `shard`: se quitan las filas anteriores a 2026 y se deja **una fila por
  `company + title + location`**, la de su primera aparición.

`row_number()` numera las copias de cada oferta por fecha; nos quedamos con la número 1.

In [10]:
duckdb.sql("""
    CREATE OR REPLACE TABLE limpio AS
    SELECT * FROM jobs WHERE source <> 'shard'
    UNION ALL
    SELECT * EXCLUDE (rn) FROM (
        SELECT *,
               row_number() OVER (PARTITION BY company, title, location
                                  ORDER BY first_seen) AS rn
        FROM jobs
        WHERE source = 'shard' AND first_seen >= '2026-01-01'
    )
    WHERE rn = 1
""")

duckdb.sql("""
    SELECT (SELECT count(*) FROM jobs)   AS filas_original,
           (SELECT count(*) FROM limpio) AS filas_limpio
""").df()

,filas_original,filas_limpio
0,11462654,5935313


### ¿Cambia la inflación de experiencia?

In [11]:
duckdb.sql("""
    SELECT 'original' AS tabla, count(*) AS con_experiencia,
           count(*) FILTER (WHERE years_experience >= 3) AS piden_3_o_mas,
           round(100 * piden_3_o_mas / con_experiencia, 2) AS pct
    FROM jobs
    WHERE level = 'junior' AND years_experience IS NOT NULL AND years_experience <> 18
    UNION ALL
    SELECT 'limpio', count(*),
           count(*) FILTER (WHERE years_experience >= 3),
           round(100 * count(*) FILTER (WHERE years_experience >= 3) / count(*), 2)
    FROM limpio
    WHERE level = 'junior' AND years_experience IS NOT NULL AND years_experience <> 18
""").df()

,tabla,con_experiencia,piden_3_o_mas,pct
0,original,38118,2868,7.52
1,limpio,38118,2868,7.52


### ¿Cambia el número de ofertas junior?

In [12]:
duckdb.sql("""
    SELECT 'original' AS tabla, source = 'shard' AS es_shard, count(*) AS ofertas_junior
    FROM jobs WHERE level IN ('junior', 'intern') GROUP BY 2
    UNION ALL
    SELECT 'limpio', source = 'shard', count(*)
    FROM limpio WHERE level IN ('junior', 'intern') GROUP BY 2
    ORDER BY tabla DESC, es_shard
""").df()

,tabla,es_shard,ofertas_junior
0,original,False,434333
1,original,True,137396
2,limpio,False,434333
3,limpio,True,34888


**Lectura:**
- El dataset pasa de **11,46 M a 5,94 M filas** (−48 %).
- La **inflación no cambia (7,5 %)**: las filas de `shard` nunca traen `years_experience`, así que
  la cifra ya se calculaba solo con las otras fuentes. ✔️ Conclusión robusta.
- El **número de ofertas junior sí cambia**: `shard` pasa de 137 k a 35 k. Sin deduplicar,
  las empresas de `shard` parecerían contratar ~4 veces más juniors de lo que contratan.
  Esto importa para la pregunta *¿quién contrata juniors de verdad?*

## 7. Ubicación: menos fiable de lo que parece

In [13]:
duckdb.sql("""
    SELECT location, count(*) AS filas
    FROM limpio
    GROUP BY location
    ORDER BY filas DESC
    LIMIT 12
""").df()

,location,filas
0,United States,1413348
1,NaN,189456
2,Germany,107841
3,United Kingdom,66809
4,"Nan, Nan, Nan",60941
5,Client Careers Site,49967
6,India,45813
7,Canada,35128
8,2 Locations,35032
9,Australia,32819


**Lectura:** muchas ubicaciones no son ciudades:
- solo el país (`United States`, `Germany`, `United Kingdom`),
- el **texto literal** `Nan, Nan, Nan` (no es un nulo de verdad, es una cadena),
- nombres del portal de empleo (`Client Careers Site`), recuentos (`2 Locations`) o
  códigos internos (`DE300`).

→ Para análisis geográfico, usar **`country`** (más limpio) y convertir `Nan, Nan, Nan` en nulo.
→ Limitación de la regla de duplicados: con estas ubicaciones, plazas de tiendas distintas
  parecen la misma oferta. Por eso solo deduplicamos `shard`.

## 8. URL: no siempre identifica una oferta

In [14]:
duckdb.sql("""
    SELECT url, count(*) AS filas
    FROM jobs
    WHERE url IS NOT NULL
    GROUP BY url
    HAVING count(*) > 1
    ORDER BY filas DESC
    LIMIT 5
""").df()

,url,filas
0,https://chery.zhiye.com/social/jobs,1162
1,https://dreame.zhiye.com/social/jobs,1127
2,https://leapmotor.zhiye.com/social/jobs,879
3,https://boe.zhiye.com/social/jobs,313
4,https://app.dataannotation.tech/worker_signup?projects=PROG_SA&worker_src=L&...,200


**Lectura:** algunas empresas (sobre todo portales chinos `*.zhiye.com`) ponen la **misma URL
genérica** en cientos de ofertas, y 857 k filas no tienen URL. → La URL **no sirve como
identificador único** de oferta.

## 9. Columnas categóricas

In [15]:
for col in ["employment", "remote", "country", "category"]:
    display(duckdb.sql(f"""
        SELECT {col}, count(*) AS filas,
               round(100 * count(*) / sum(count(*)) OVER (), 1) AS pct
        FROM limpio
        GROUP BY {col}
        ORDER BY filas DESC
        LIMIT 8
    """).df())

,employment,filas,pct
0,NaN,3133909,52.8
1,full_time,2362101,39.8
2,part_time,288829,4.9
3,internship,79418,1.3
4,contract,71056,1.2


,remote,filas,pct
0,False,5441427,91.7
1,True,493886,8.3


,country,filas,pct
0,United States,2995107,50.5
1,NaN,1633149,27.5
2,Germany,259885,4.4
3,United Kingdom,162117,2.7
4,India,120538,2.0
5,Canada,119687,2.0
6,France,81346,1.4
7,Australia,66348,1.1


,category,filas,pct
0,other,1406524,23.7
1,healthcare,535968,9.0
2,engineering,534618,9.0
3,retail,478051,8.1
4,sales,452344,7.6
5,operations,362945,6.1
6,administrative,336020,5.7
7,finance,303654,5.1


**Lectura (sobre la tabla limpia):**
- `employment` está vacío en **más de la mitad** de las filas (53 %) → poco útil como variable.
- `remote`: solo un 8 % en remoto.
- `country`: la mitad es EE. UU. y un **27 % está vacío**.
- `category`: la clase `other` es la mayor (**24 %**), así que los análisis por sector dejan
  fuera una cuarta parte de las ofertas.

Nada bloquea el proyecto, pero todo esto va a la sección de limitaciones del README.

## 10. Salario (para la Fase 3)

In [16]:
duckdb.sql("""
    SELECT pay_currency, count(*) AS filas,
           median(pay_min) AS mediana_min, median(pay_max) AS mediana_max,
           min(pay_min) AS minimo, max(pay_max) AS maximo
    FROM limpio
    WHERE pay_min IS NOT NULL OR pay_max IS NOT NULL
    GROUP BY pay_currency
    ORDER BY filas DESC
    LIMIT 8
""").df()

,pay_currency,filas,mediana_min,mediana_max,minimo,maximo
0,USD,245681,70000.0,90000.0,0,160000000
1,GBP,6087,50000.0,60000.0,10,750000
2,SGD,4050,3500.0,6000.0,1,495000
3,EUR,4050,51194.0,68000.0,4,1000000
4,CNY,2226,25.0,50.0,1,1600000
5,CAD,2142,100000.0,128300.0,1,999999
6,IDR,621,3000000.0,4000000.0,30000,45000000
7,AUD,519,96836.0,112401.0,28,465000


In [17]:
duckdb.sql("""
    SELECT count(*) FILTER (WHERE pay_currency = 'USD' AND pay_max < 1000)  AS usd_probablemente_por_hora,
           count(*) FILTER (WHERE pay_currency = 'USD' AND pay_max >= 1000) AS usd_probablemente_anual,
           count(*) FILTER (WHERE pay_min > pay_max)                        AS min_mayor_que_max
    FROM limpio
""").df()

,usd_probablemente_por_hora,usd_probablemente_anual,min_mayor_que_max
0,34451,211230,261


**Lectura:** el salario mezcla **monedas** y **periodos** (en USD hay muchos valores < 1.000 que
casi seguro son por hora), tiene extremos absurdos (máximos de millones) y casos con
`pay_min > pay_max`. Antes del modelo de salario habrá que: quedarse con USD, separar
hora/año (o anualizar), y recortar extremos.

## 11. Skills

In [18]:
duckdb.sql("""
    SELECT trim(skill) AS skill, count(*) AS ofertas
    FROM (SELECT unnest(string_split(skills, ';')) AS skill FROM limpio WHERE skills IS NOT NULL)
    GROUP BY 1
    ORDER BY ofertas DESC
    LIMIT 15
""").df()

,skill,ofertas
0,customer_service,182924
1,python,67190
2,talent_acquisition,66487
3,ms_office,50952
4,project_management,45551
5,aws,45326
6,ml,44581
7,sql,43197
8,patient_care,41349
9,crm,40764


**Lectura:** las skills vienen **normalizadas** (`python`, `ms_office`, `customer_service`), lo
que facilitará el skill gap (Fase 4). `string_split` + `unnest` convierte la lista separada
por `;` en una fila por skill.

## Resumen de decisiones para el script de limpieza

| # | Problema | Decisión |
|---|---|---|
| 1 | `posted` con fechas rotas | Leer como `VARCHAR`; usar `first_seen` como fecha |
| 2 | `years_experience = 18` es la edad mínima | Tratarlo como nulo |
| 3 | `shard` guarda una copia de cada oferta cada 2 días | Deduplicar `shard` por `company + title + location`, quedándose con la primera aparición |
| 4 | `shard` con fechas 2020-2023, sin URL ni ubicación | Excluir (`first_seen < 2026-01-01`) |
| 5 | `shard` nunca registra cierres | No usar `shard` para análisis de cierre ni de ofertas fantasma |
| 6 | `ghost_score` solo existe para ofertas abiertas | Analizar fantasmas solo con ofertas abiertas de fuentes que no sean `shard` |
| 7 | `location = 'Nan, Nan, Nan'` y ubicaciones que no son ciudades | Convertir el texto `Nan, Nan, Nan` en nulo; usar `country` para análisis geográfico |
| 8 | Las demás fuentes no se deduplican | Limitación: republicaciones como la de Gopuff en Birmingham se cuentan dos veces (efecto pequeño) |
| 9 | Salario con monedas, periodos y extremos mezclados | Se trata en la Fase 3 |

**Cifras clave:** 11,46 M filas → **5,94 M** tras limpiar. La inflación junior se mantiene en
**7,5 %**. Las ofertas junior de `shard` bajan de 137 k a 35 k.